# ⚡ Manga Studio Cloud (Google Colab GPU)
### Execute o Manga Studio gratuitamente na nuvem da Google!
Todo o processamento pesado de imagens e OCR roda nos servidores da Google com **0% de uso do seu PC**.

---
### 🚀 Como usar:
1. No menu superior, vá em **Ambiente de execução** > **Alterar tipo de ambiente de execução** e escolha **T4 GPU** (acelera o OCR).
2. Clique no botão de Play `[ ▶ ]` na célula abaixo.
3. Em cerca de 30 segundos, um link público seguro (ex: `https://xxxx.trycloudflare.com`) será exibido.
4. Abra o link no seu navegador (no computador ou celular) para usar o **Manga Studio completo**!

In [ ]:
# @title ▶ Iniciar Manga Studio na Nuvem (Clique no Play)
import os, subprocess, time, re, sys

# 1. Garante que o diretório de trabalho inicial seja /content para evitar erros de pasta deletada
os.chdir('/content')

print("Atualizando / Baixando o Manga Studio do GitHub...")
if os.path.exists('/content/manga-translator/.git'):
    !git -C /content/manga-translator pull
else:
    !rm -rf /content/manga-translator
    !git clone https://github.com/Jos9011/SKILL-TRADUCAO-MANGA.git /content/manga-translator

%cd /content/manga-translator

print("Verificando dependências de OCR e tradução...")
!pip install -q flask pillow opencv-python-headless rapidocr_onnxruntime deep_translator pyphen wordninja

print("Configurando túnel seguro Cloudflare...")
!which cloudflared > /dev/null 2>&1 || (wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1)

# Encerra qualquer processo anterior para liberar a porta 5050
!pkill -f 'studio/server.py' || true
!pkill -f 'cloudflared tunnel' || true
time.sleep(1)

# Inicia o servidor Flask em segundo plano
p_srv = subprocess.Popen([sys.executable, 'studio/server.py'])
time.sleep(3)

# Inicia túnel Cloudflare para gerar o link público
p_tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5050'], stderr=subprocess.PIPE, text=True)

print('\n' + '='*60)
print('   🚀 MANGA STUDIO INICIANDO NA NUVEM DA GOOGLE...')
print('='*60)

for line in p_tunnel.stderr:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        print(f'\n✅ MANGA STUDIO PRONTO E OPERACIONAL!')
        print(f'👉 LINK DE ACESSO: {url}')
        print('\n(Você pode abrir este link no computador ou celular. Deixe esta aba do Colab aberta enquanto traduz.)\n')
        break


In [ ]:
# @title 📁 (Opcional) Conectar ao seu Google Drive
# Execute esta célula se quiser pegar mangás diretamente do seu Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive conectado com sucesso! Suas pastas estão em /content/drive/MyDrive/')
